# Dự báo Doanh số Thương mại Điện tử
## Giai đoạn 4 — Random Forest Regressor (Ensemble: Bagging)

> **Lưu ý phụ thuộc:** Notebook này yêu cầu đã chạy xong **GĐ1.ipynb** và **GĐ3.ipynb** để có sẵn:
> - Các hàm metric: `calculate_mse`, `calculate_rmse`, `calculate_mae`, `calculate_r2`
> - Class `DanhGiaMoHinh` và object `my_evaluator`
> - Class `CustomStandardScaler` và dữ liệu `X_train_sc`, `X_test_sc`, `y_train`, `y_test`
> - Class `Node` và `DecisionTreeRegressor` (nền tảng bắt buộc cho Random Forest)
>
> **Mục tiêu:** Áp dụng kỹ thuật **Bagging** (Bootstrap Aggregating) — huấn luyện song song N cây Decision Tree
> trên các tập dữ liệu Bootstrap khác nhau, sau đó tổng hợp kết quả bằng trung bình cộng.
> Mục đích là giảm **Variance** và khắc phục **Overfitting** của một Decision Tree đơn lẻ.

---

## Import thư viện

In [1]:
import numpy as np
import pandas as pd

## Nạp lại từ GĐ1 & GĐ3

Notebook này giả định đã chạy GĐ1 và GĐ3 trong cùng kernel để có sẵn các class và hàm.

---
## Load & Chuẩn hóa Dữ liệu

In [4]:
# Load dữ liệu đã EDA từ Phần I — BẮT BUỘC dùng pd.read_csv()
X_train = pd.read_csv("../dataset_ready/X_train.csv").values.astype(float)
X_test  = pd.read_csv("../dataset_ready/X_test.csv").values.astype(float)
y_train = pd.read_csv("../dataset_ready/y_train.csv").values.ravel().astype(float)
y_test  = pd.read_csv("../dataset_ready/y_test.csv").values.ravel().astype(float)

# Kiểm tra NaN
for name, arr in [("X_train", X_train), ("X_test", X_test),
                   ("y_train", y_train), ("y_test", y_test)]:
    assert not np.isnan(arr).any(), f"{name} có giá trị NaN!"

print(f"X_train : {X_train.shape}  |  y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}   |  y_test  : {y_test.shape}")

# Chuẩn hóa — fit chỉ trên train, transform cả hai
scaler     = CustomStandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

assert np.allclose(X_train_sc.mean(axis=0), 0, atol=1e-6), "Mean không ≈ 0!"
assert np.allclose(X_train_sc.std(axis=0),  1, atol=1e-6), "Std không ≈ 1!"
print("\nDữ liệu hợp lệ và đã chuẩn hóa xong.")

X_train : (487, 9)  |  y_train : (487,)
X_test  : (122, 9)   |  y_test  : (122,)

Dữ liệu hợp lệ và đã chuẩn hóa xong.


---
## Giai đoạn 4 — Class `RandomForestRegressor`

### Lý thuyết Bagging

**Vấn đề:** Một Decision Tree đơn lẻ có **Variance cao** — kết quả thay đổi nhiều theo tập huấn luyện (Overfitting).

**Giải pháp — Bagging (Bootstrap Aggregating):**
1. **Bootstrap:** Tạo N tập huấn luyện con bằng cách lấy mẫu ngẫu nhiên **có hoàn lại** (with replacement) từ tập gốc — mỗi tập con cùng kích thước với tập gốc.
2. **Parallel Training:** Huấn luyện N cây `DecisionTreeRegressor` độc lập, mỗi cây trên một tập Bootstrap riêng. Đồng thời áp dụng **Feature Subsampling** (`max_features`) để các cây đa dạng hơn.
3. **Aggregation:** Dự báo cuối cùng = trung bình cộng của dự báo từ tất cả các cây.

$$\hat{y} = \frac{1}{N} \sum_{i=1}^{N} T_i(X)$$

**Tại sao hiệu quả?** Trung bình hóa nhiều mô hình có **tương quan thấp** với nhau sẽ làm giảm Variance mà không tăng Bias đáng kể.

In [5]:
class RandomForestRegressor:
    """
    Random Forest Regressor — Ensemble học theo phương pháp Bagging.

    Cơ chế:
    - Huấn luyện N cây DecisionTreeRegressor song song trên các tập Bootstrap khác nhau.
    - Mỗi cây chỉ xét một tập con ngẫu nhiên các Feature tại mỗi lần split (max_features).
    - Dự báo = trung bình cộng của toàn bộ N cây.

    Parameters
    ----------
    n_estimators   : int   — Số lượng cây trong rừng (mặc định: 100)
    max_depth      : int   — Chiều sâu tối đa của mỗi cây (mặc định: 5)
    min_samples_split : int — Số mẫu tối thiểu để split một node (mặc định: 2)
    max_features   : int | None — Số feature ngẫu nhiên xét tại mỗi split.
                                  None = dùng tất cả feature (giống Decision Tree thuần).
                                  Thông thường đặt = int(sqrt(n_features)) hoặc n_features // 3.
    random_state   : int | None — Seed để tái tạo kết quả (mặc định: None)
    evaluator      : DanhGiaMoHinh | None — Object đánh giá từ GĐ1
    """

    def __init__(
        self,
        n_estimators=100,
        max_depth=5,
        min_samples_split=2,
        max_features=None,
        random_state=None,
        evaluator=None
    ):
        self.n_estimators      = n_estimators
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.max_features      = max_features
        self.random_state      = random_state
        self.evaluator         = evaluator
        self.trees             = []   # danh sách các DecisionTreeRegressor đã huấn luyện

    # ──────────────────────────────────────────────────────────────────────────
    # PRIVATE: Bootstrap Sampling
    # ──────────────────────────────────────────────────────────────────────────
    def _bootstrap(self, X, y):
        """
        Lấy mẫu ngẫu nhiên có hoàn lại (Bootstrapping).

        Từ tập (X, y) gốc gồm n mẫu, rút ngẫu nhiên n chỉ số *có hoàn lại*.
        Trung bình ~63.2% mẫu gốc xuất hiện trong tập Bootstrap;
        ~36.8% còn lại (Out-of-Bag) có thể dùng để validate nội bộ.

        Parameters
        ----------
        X : array (n_samples, n_features)
        y : array (n_samples,)

        Returns
        -------
        X_boot : array (n_samples, n_features)  — tập Bootstrap
        y_boot : array (n_samples,)             — nhãn tương ứng
        """
        n_samples = X.shape[0]
        indices   = np.random.choice(n_samples, size=n_samples, replace=True)
        return X[indices], y[indices]

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC: Huấn luyện
    # ──────────────────────────────────────────────────────────────────────────
    def fit(self, X, y):
        """
        Huấn luyện Random Forest:
        1. Đặt random seed (nếu có) để đảm bảo tái tạo được kết quả.
        2. Lặp n_estimators lần:
           a. Tạo tập Bootstrap từ (X, y).
           b. Khởi tạo và huấn luyện một DecisionTreeRegressor trên tập đó.
           c. Lưu cây đã huấn luyện vào self.trees.

        Parameters
        ----------
        X : array (n_samples, n_features)
        y : array (n_samples,)

        Returns
        -------
        self
        """
        if self.random_state is not None:
            np.random.seed(self.random_state)

        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        self.trees = []

        # Xác định max_features mặc định nếu không truyền vào
        n_features       = X.shape[1]
        effective_max_f  = self.max_features if self.max_features is not None \
                           else n_features

        for i in range(self.n_estimators):
            # Bước 1: Bootstrap
            X_boot, y_boot = self._bootstrap(X, y)

            # Bước 2: Xây dựng cây — truyền max_features để random feature subsampling
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=effective_max_f
            )
            tree.fit(X_boot, y_boot)

            # Bước 3: Lưu cây vào danh sách
            self.trees.append(tree)

        return self

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC: Dự báo
    # ──────────────────────────────────────────────────────────────────────────
    def predict(self, X):
        """
        Dự báo bằng cách lấy trung bình cộng dự báo của toàn bộ N cây.

        Cơ chế:
            predictions[i] = T_i(X)  cho mỗi cây thứ i
            y_pred = mean(predictions, axis=0)

        Parameters
        ----------
        X : array (n_samples, n_features)

        Returns
        -------
        y_pred : array (n_samples,)
        """
        if not self.trees:
            raise RuntimeError("Cần gọi fit() trước khi predict()")

        X = np.asarray(X, dtype=float)
        # Thu thập dự báo từ tất cả các cây: shape (n_estimators, n_samples)
        all_predictions = np.array([tree.predict(X) for tree in self.trees])
        # Aggregation: trung bình theo chiều cây
        return np.mean(all_predictions, axis=0)

    # ──────────────────────────────────────────────────────────────────────────
    # PUBLIC: Đánh giá
    # ──────────────────────────────────────────────────────────────────────────
    def evaluate(self, X_test, y_test, model_name="Random Forest"):
        """
        Dự báo và đánh giá — gọi DanhGiaMoHinh từ GĐ1, không tự tính metric.

        Returns
        -------
        dict: {'MSE', 'RMSE', 'MAE', 'R2'}
        """
        if self.evaluator is None:
            raise RuntimeError("evaluator chưa được inject. Truyền DanhGiaMoHinh() vào __init__.")
        y_pred = self.predict(X_test)
        return self.evaluator.evaluate_all(y_test, y_pred, model_name=model_name)


print("Đã định nghĩa xong class RandomForestRegressor.")

Đã định nghĩa xong class RandomForestRegressor.


---
## Kiểm tra nhanh với dữ liệu đơn giản

Dùng bài toán đồ chơi để xác minh cơ chế Bootstrap + Aggregation hoạt động đúng trước khi chạy dữ liệu thực.

In [6]:
# ── Bài toán đồ chơi ─────────────────────────────────────────────────────────
#   X < 5 → y ≈ 10 | X >= 5 → y ≈ 20
#   Random Forest phải cho R² ≈ 1.0 trên tập dữ liệu tách biệt hoàn toàn.

X_toy = np.array([[1],[2],[3],[4],[6],[7],[8],[9]], dtype=float)
y_toy = np.array([10, 10, 10, 10, 20, 20, 20, 20], dtype=float)

rf_toy = RandomForestRegressor(
    n_estimators=10,
    max_depth=3,
    min_samples_split=2,
    random_state=42,
    evaluator=my_evaluator
)
rf_toy.fit(X_toy, y_toy)
y_toy_pred = rf_toy.predict(X_toy)

# Kiểm tra số lượng cây
assert len(rf_toy.trees) == 10, f"Phải có đúng 10 cây, nhận được: {len(rf_toy.trees)}"

# Dữ liệu tách biệt hoàn toàn → R² phải rất gần 1.0
r2_toy = calculate_r2(y_toy, y_toy_pred)
assert r2_toy > 0.95, f"R² đồ chơi phải > 0.95, nhận được: {r2_toy:.4f}"

# Kiểm tra shape output
assert y_toy_pred.shape == y_toy.shape, "Shape dự báo không khớp!"

print(f"Số cây trong rừng : {len(rf_toy.trees)}")
print(f"R² đồ chơi        : {r2_toy:.4f}")
print(f"Dự báo mẫu        : {y_toy_pred}")
print("Tất cả assert kiểm tra nhanh PASSED.")

Số cây trong rừng : 10
R² đồ chơi        : 0.9800
Dự báo mẫu        : [10. 10. 10. 12. 20. 20. 20. 20.]
Tất cả assert kiểm tra nhanh PASSED.


---
## Khảo sát Hyperparameter

Quan sát ảnh hưởng của `n_estimators` và `max_features` đến hiệu năng mô hình.

In [7]:
# ── Khảo sát n_estimators ─────────────────────────────────────────────────────
# Kỳ vọng: R² Test tăng dần khi thêm cây, rồi ổn định (không tăng mãi mãi).
n_features = X_train_sc.shape[1]
default_max_f = max(1, int(np.sqrt(n_features)))   # sqrt(n_features) là mặc định phổ biến

print("── Khảo sát n_estimators (max_features=sqrt(n_features)) ──")
print(f"{'N Trees':>8} | {'R² Train':>10} | {'R² Test':>10} | {'RMSE Test':>12}")
print("-" * 48)

for n_est in [10, 30, 50, 100]:
    rf = RandomForestRegressor(
        n_estimators=n_est,
        max_depth=7,
        min_samples_split=5,
        max_features=default_max_f,
        random_state=42,
        evaluator=my_evaluator
    )
    rf.fit(X_train_sc, y_train)
    r2_train  = calculate_r2(y_train,  rf.predict(X_train_sc))
    r2_test   = calculate_r2(y_test,   rf.predict(X_test_sc))
    rmse_test = calculate_rmse(y_test, rf.predict(X_test_sc))
    print(f"{n_est:>8} | {r2_train:>10.4f} | {r2_test:>10.4f} | {rmse_test:>12.4f}")

── Khảo sát n_estimators (max_features=sqrt(n_features)) ──
 N Trees |   R² Train |    R² Test |    RMSE Test
------------------------------------------------
      10 |     0.8875 |     0.5111 |    7859.8312
      30 |     0.9140 |     0.6224 |    6907.9587
      50 |     0.9052 |     0.6290 |    6846.5662
     100 |     0.9041 |     0.6493 |    6656.7455


In [8]:
# ── Khảo sát max_features ─────────────────────────────────────────────────────
# max_features kiểm soát mức độ tương quan giữa các cây:
#   - Nhỏ hơn → cây đa dạng hơn → Variance thấp hơn, nhưng Bias có thể tăng.
#   - Bằng n_features → giống Bagging thuần (không random feature).

print("── Khảo sát max_features (n_estimators=50, max_depth=7) ──")
print(f"{'max_feat':>9} | {'R² Train':>10} | {'R² Test':>10} | {'RMSE Test':>12}")
print("-" * 50)

for mf_label, mf_val in [
    (f"1          ",  1),
    (f"sqrt={default_max_f}   ", default_max_f),
    (f"n//2={n_features//2}  ",  n_features // 2),
    (f"all={n_features}    ",  n_features),
]:
    rf = RandomForestRegressor(
        n_estimators=50,
        max_depth=7,
        min_samples_split=5,
        max_features=mf_val,
        random_state=42,
        evaluator=my_evaluator
    )
    rf.fit(X_train_sc, y_train)
    r2_train  = calculate_r2(y_train,  rf.predict(X_train_sc))
    r2_test   = calculate_r2(y_test,   rf.predict(X_test_sc))
    rmse_test = calculate_rmse(y_test, rf.predict(X_test_sc))
    print(f"{mf_label:>9} | {r2_train:>10.4f} | {r2_test:>10.4f} | {rmse_test:>12.4f}")

── Khảo sát max_features (n_estimators=50, max_depth=7) ──
 max_feat |   R² Train |    R² Test |    RMSE Test
--------------------------------------------------
1           |     0.7867 |     0.4977 |    7967.1266
sqrt=3    |     0.9052 |     0.6290 |    6846.5662
 n//2=4   |     0.9130 |     0.6290 |    6846.5691
all=9     |     0.9301 |     0.7045 |    6110.5414


---
## Đánh giá chính thức & So sánh với Decision Tree đơn lẻ

In [9]:
print("=" * 50)

# ── Huấn luyện Random Forest chính thức ──────────────────────────────────────
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=7,
    min_samples_split=5,
    max_features=default_max_f,
    random_state=42,
    evaluator=my_evaluator
)
rf_model.fit(X_train_sc, y_train)
rf_results = rf_model.evaluate(X_test_sc, y_test, model_name="Random Forest (100 cây, depth=7)")

print()

# ── So sánh với Decision Tree đơn lẻ (depth=5, từ GĐ3) ──────────────────────
dt_baseline = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=10,
    evaluator=my_evaluator
)
dt_baseline.fit(X_train_sc, y_train)
dt_results = dt_baseline.evaluate(X_test_sc, y_test, model_name="Decision Tree (depth=5) [Baseline]")

print()
print("── Bảng So sánh Tổng hợp ──")
print(f"{'Mô hình':<40} | {'R² Test':>8} | {'RMSE Test':>12}")
print("-" * 66)
print(f"{'Decision Tree (depth=5)':<40} | {dt_results['R2']:>8.4f} | {dt_results['RMSE']:>12.4f}")
print(f"{'Random Forest (100 cây, depth=7)':<40} | {rf_results['R2']:>8.4f} | {rf_results['RMSE']:>12.4f}")

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert rf_results["R2"] > 0, f"R² âm — mô hình tệ hơn baseline mean: {rf_results['R2']:.4f}"
assert len(rf_model.trees) == 100, "Số cây không đúng!"
assert rf_model.predict(X_test_sc).shape == y_test.shape, "Shape dự báo sai!"

# Kỳ vọng RF tốt hơn hoặc tương đương DT đơn lẻ
improvement = rf_results["R2"] - dt_results["R2"]
print(f"\nCải thiện R² so với Decision Tree: {improvement:+.4f}")
print(f"\nSanity check R² > 0: PASSED ({rf_results['R2']:.4f})")
print("\nrf_model sẵn sàng để so sánh ở Giai đoạn 6 (Pipeline & Evaluation).")

── Đánh giá Random Forest (100 cây, depth=7) ──
MSE  : 44312261.1472
RMSE : 6656.7455
MAE  : 5061.1352
R²   : 0.6493

── Đánh giá Decision Tree (depth=5) [Baseline] ──
MSE  : 62879391.9263
RMSE : 7929.6527
MAE  : 6036.6080
R²   : 0.5024

── Bảng So sánh Tổng hợp ──
Mô hình                                  |  R² Test |    RMSE Test
------------------------------------------------------------------
Decision Tree (depth=5)                  |   0.5024 |    7929.6527
Random Forest (100 cây, depth=7)         |   0.6493 |    6656.7455

Cải thiện R² so với Decision Tree: +0.1469

Sanity check R² > 0: PASSED (0.6493)

rf_model sẵn sàng để so sánh ở Giai đoạn 6 (Pipeline & Evaluation).
